# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform basic data wrangling on the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library in a reproducible and programmatic manner.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore summary information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic metadata
md = dataset.metadata
print(f"Name: {md.name}\n\nDescription: {md.description}\n\nLicense: {md.license}\n\nVersion: {md.version}\nPublished: {md.datePublished}")

## 2. Data Overview
List all available record sets in the dataset, their corresponding `@id`s, and summarize the available fields/columns for each.

In [ ]:
# List record sets, fields, and columns by @id
print("Available RecordSets and their fields/columns:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for f in rs.fields:
        col_ids = [c.id for c in getattr(f, 'columns', [])]
        print(f"    - {f.name:40} (@id: {f.id}) columns: {col_ids}")

**Choose a RecordSet by its `@id` for further exploration—see above for available options. For this example, we'll use the main experimental RecordSet.**

In [ ]:
# Identify the primary record set (by inspection above)
# For this dataset, the canonical main RecordSet often contains protein data.
# You may need to adjust the @id if the printed output is updated.
main_record_set_id = None
for rs in record_sets:
    if 'protein' in rs.name.lower() or 'main' in rs.name.lower() or 'extracellular' in rs.name.lower():
        main_record_set_id = rs.id
if not main_record_set_id:
    main_record_set_id = record_sets[0].id  # fallback: use first
print(f"Using record set @id: {main_record_set_id}")

# Show a sample record in this record set
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data from each record set into Pandas DataFrames. We'll focus subsequent analysis on the main experimental record set by its `@id`.

In [ ]:
# Extract all record sets into DataFrames by @id
dataframes = dict()
for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for RecordSet @id {rs.id} with {len(df)} rows and columns: {df.columns.tolist()}")

# Focus on the main record set
main_df = dataframes[main_record_set_id]
print("\nFirst few rows of the main record set DataFrame:")
display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field by its `@id` and run some typical EDA steps: filtering on a value, normalizing the data, and grouping by another field.

<span style="color:gray">*(Replace the following field IDs according to the printed overview!)*</span>

In [ ]:
# Select a numeric field @id from your record set, e.g., 'abundance', 'MW', or similar
# For illustration, we'll search for a numeric column containing 'abundance' or 'MW' (molecular weight).
import numpy as np

numeric_field_id = None
for col in main_df.columns:
    lowered = col.lower()
    if 'abundance' in lowered or 'mw' in lowered or 'peptide' in lowered:
        numeric_field_id = col
        break

if not numeric_field_id:
    numeric_field_id = main_df.select_dtypes(include=[np.number]).columns[0]  # fallback

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Threshold for filtering
threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype == float or main_df[numeric_field_id].dtype == int else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.1f} (n={len(filtered_df)})")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} (mean=0, std=1):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field, e.g. sample/treatment
group_field_id = None
for col in main_df.columns:
    if 'sample' in col.lower() or 'treatment' in col.lower() or 'group' in col.lower():
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No group field found; skipping groupby.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, where possible, compare it across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if available)
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Croissant dataset using standard, FAIR-compliant tooling (`mlcroissant`).
- We identified the main record set and examined its schema by `@id`.
- Data was loaded into a DataFrame and basic filtering, normalization, and grouping were demonstrated using numeric and categorical fields referenced by their `@id`s.
- Initial visualizations provide insight into the data's distribution and possible group-level effects for further biological/statistical analysis.

> **Note:** For reproducible or domain-specific work, ensure you cross-validate field selection by `@id` from the schema printout and consult the original dataset documentation for proper interpretation of field values.